# Wikidata Agent Evaluation

This notebook evaluates a Python AI agent built with LangChain that answers questions by querying Wikidata. This agent uses Qwen 2.5 7B 4-bit to intelligently search for entities/properties and construct SPARQL queries.

## Features

* Search for entities and properties in Wikidata
* Execute SPARQL queries against Wikidata's SPARQL endpoint
* Generate SPARQL queries from natural language questions
* Evaluate agent's performance against test datasets

## 1. Install Required Packages

In [ ]:
!pip install langchain>=0.1.0 langchain-core>=0.1.0 SPARQLWrapper>=2.0.0 python-dotenv>=1.0.0 requests>=2.31.0 pandas tqdm
!pip install langchain-community transformers accelerate bitsandbytes

## 2. Download Test Dataset

In [ ]:
import os

# Create dataset directory if it doesn't exist
!mkdir -p dataset/qald_9_plus

# Download test dataset
!wget -O dataset/qald_9_plus/qald_9_plus_test_wikidata.json https://raw.githubusercontent.com/gansixeneh/FROG-2.0/refs/heads/version_2.2.eval/dataset/qald_9_plus/qald_9_plus_test_wikidata.json

# Verify download
!ls -la dataset/qald_9_plus/

## 3. Define Tools for Wikidata Agent

In [ ]:
from typing import Optional, List, Literal, Dict, Any, ClassVar, Tuple
from langchain_core.tools import BaseTool
import requests
from SPARQLWrapper import SPARQLWrapper, JSON

class SearchWikidataTool(BaseTool):
    name: str = "search_entity_property"
    description: str = """Search for entities or properties in Wikidata by name or label.
    
    Args:
        term: The search term to look for in Wikidata
        type: Type of search - must be either "entity" or "property"
        limit: Maximum number of results to return (default: 5)
    
    Returns:
        A list of matching entities or properties with their details (id, label, description)
    """
    
    def _run(self, term: str, type: Literal["entity", "property"] = "entity", limit: int = 5) -> List[Dict[str, Any]]:
        """Search for entities or properties in Wikidata"""
        if type == "entity":
            return self._search_entity(term, limit)
        elif type == "property":
            return self._search_property(term, limit)
        else:
            raise ValueError(f"Invalid search type: {type}. Must be 'entity' or 'property'")
    
    def _search_entity(self, term: str, limit: int) -> List[Dict[str, Any]]:
        url = "https://www.wikidata.org/w/api.php"
        params = {
            "action": "wbsearchentities",
            "format": "json",
            "language": "en",
            "search": term,
            "limit": limit
        }
        
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        results = []
        for item in data.get("search", []):
            result = {
                "id": item.get("id"),
                "label": item.get("label", ""),
                "description": item.get("description", ""),
                "url": item.get("url", "")
            }
            results.append(result)
            
        return results
    
    def _search_property(self, term: str, limit: int) -> List[Dict[str, Any]]:
        url = "https://www.wikidata.org/w/api.php"
        params = {
            "action": "wbsearchentities",
            "format": "json",
            "language": "en",
            "search": term,
            "type": "property",
            "limit": limit
        }
        
        response = requests.get(url, params=params)
        response.raise_for_status()
        data = response.json()
        
        results = []
        for item in data.get("search", []):
            result = {
                "id": item.get("id"),
                "label": item.get("label", ""),
                "description": item.get("description", ""),
                "url": item.get("url", "")
            }
            results.append(result)
            
        return results

class ExecuteSPARQLTool(BaseTool):
    name: str = "execute_sparql"
    description: str = """Execute a SPARQL query against Wikidata.
    
    Args:
        query: The complete SPARQL query string to execute
        limit: Maximum number of results to return (default: 5)
    
    Returns:
        The query results or error information if the query fails
    """
    
    def __init__(self):
        super().__init__()
        self._endpoint = "https://query.wikidata.org/sparql"
        self._sparql = SPARQLWrapper(self._endpoint)
        self._sparql.setReturnFormat(JSON)
        self._sparql.addCustomHttpHeader("User-Agent", "LangChain Wikidata Agent/1.0")
        
    def _run(self, query: str, limit: int = 5) -> Dict[str, Any]:
        """Execute a SPARQL query against Wikidata"""
        try:
            limit_value = int(limit)
            
            if "LIMIT" not in query.upper():
                query += f" LIMIT {limit_value}"
            
            self._sparql.setQuery(query)
            results = self._sparql.query().convert()
            
            processed_results = []
            
            if "results" in results and "bindings" in results["results"]:
                bindings = results["results"]["bindings"]
                
                for binding in bindings:
                    processed_binding = {}
                    for key, value in binding.items():
                        processed_binding[key] = value.get("value", "")
                    processed_results.append(processed_binding)
                
                return {
                    "success": True,
                    "results": processed_results,
                    "count": len(processed_results),
                    "raw_results": bindings
                }
            else:
                return {
                    "success": True,
                    "results": results,
                    "count": 1
                }
        
        except Exception as e:
            return {
                "success": False,
                "error": str(e).split('\n')[0],
                "query": query
            }

## 4. Define Wikidata Agent with Qwen 2.5 7B 4-bit

In [ ]:
import os
from typing import Tuple
from langchain_core.tools import BaseTool
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import SystemMessage, HumanMessage
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_community.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline, BitsAndBytesConfig
import torch

class WikidataAgent:
    def __init__(self):
        # Initialize tools
        self.search_tool = SearchWikidataTool()
        self.sparql_tool = ExecuteSPARQLTool()
        self.tools = [self.search_tool, self.sparql_tool]

        # Create the system message with detailed instructions
        system_message = """You are an AI assistant that generates SPARQL queries for Wikidata based on user questions and the given tools.
You have access to two tools:

1. search_entity_property: Use this to search for entities or properties in Wikidata.
2. execute_sparql: Use this to run SPARQL queries against Wikidata.

To generate a SPARQL query for a user's question, you MUST follow these steps:

1. Analyze the user's question and identify the key entities and properties that need to be looked up.
2. ALWAYS use the search_entity_property tool to find the Wikidata IDs for these entities and properties. 
   NEVER guess, hallucinate, or infer entity IDs (Q-IDs) or property IDs (P-IDs).
3. For EACH entity or property in your query, show proof that you searched for it using the search_entity_property tool.
4. Construct a SPARQL query using ONLY the entity and property IDs obtained from search results.
5. You can test your query using the execute_sparql tool to verify it works.
6. If the query results are insufficient or there's an error:
   - Revise your entities/properties or try a different SPARQL query
   - Search for additional entities or properties if needed
   - Execute the new SPARQL query
7. Once you have satisfactory results, return ONLY the final SPARQL query as the response, with appropriate prefixes.

IMPORTANT: Generate SPARQL queries that ONLY return URIs WITHOUT including labels. 
Do NOT use rdfs:label, wikibase:label, or SERVICE wikibase:label in your queries.
Do NOT include variables with "Label" suffix in the SELECT clause.

CRITICAL REMINDER: 
- NEVER guess Wikidata IDs. Even if you think you know a common ID, ALWAYS verify it with search_entity_property.
- Wikidata entities start with Q (like Q42 for Douglas Adams)
- Wikidata properties start with P (like P31 for "instance of")
- Make your SPARQL queries specific and focused
- Always include relevant entity/property IDs in your SPARQL queries
- Document each search you perform so it's clear where each ID came from

Common SPARQL prefixes for Wikidata:
PREFIX wd: <http://www.wikidata.org/entity/>
PREFIX wdt: <http://www.wikidata.org/prop/direct/>
PREFIX wikibase: <http://wikiba.se/ontology#>
PREFIX p: <http://www.wikidata.org/prop/>
PREFIX ps: <http://www.wikidata.org/prop/statement/>
PREFIX pq: <http://www.wikidata.org/prop/qualifier/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX bd: <http://www.bigdata.com/rdf#>

Example process and query for "Who is the president of France?":

1. Search for "France" entity:
   > search_entity_property(term="France", type="entity")
   Result: Found Q142 (France)

2. Search for "president" property:
   > search_entity_property(term="president", type="property")
   Result: Found P35 (head of state)

3. Build and execute SPARQL query:
   ```
   SELECT ?president WHERE {
     wd:Q142 wdt:P35 ?president.
   }
   ```
"""

        # Create a prompt template with system message and human input
        self.prompt = ChatPromptTemplate.from_messages([
            SystemMessage(content=system_message),
            ("human", "{input}"),
            MessagesPlaceholder(variable_name="agent_scratchpad")
        ])

        # Initialize the Qwen model
        self._init_qwen_model()
        
        # Initialize the agent
        self.agent = create_tool_calling_agent(
            self.llm, self.tools, self.prompt
        )
        self.agent_executor = AgentExecutor.from_agent_and_tools(
            agent=self.agent,
            tools=self.tools,
            verbose=True,
            handle_parsing_errors=True,
            max_iterations=20,  # Limit number of iterations to prevent infinite loops
        )

    def _init_qwen_model(self):
        """Initialize Qwen 2.5 7B 4-bit model"""
        print("Initializing Qwen 2.5 7B 4-bit model...")
        
        # Configure 4-bit quantization
        quantization_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True
        )
        
        # Load tokenizer and model
        model_id = "Qwen/Qwen2.5-Coder-7B"
        tokenizer = AutoTokenizer.from_pretrained(model_id)
        model = AutoModelForCausalLM.from_pretrained(
            model_id,
            device_map="auto",
            quantization_config=quantization_config,
            trust_remote_code=True
        )
        
        # Create text generation pipeline
        text_generation_pipeline = pipeline(
            model=model,
            tokenizer=tokenizer,
            task="text-generation",
            max_new_tokens=512,
            temperature=0,
            repetition_penalty=1.1,
            return_full_text=False,
            trust_remote_code=True
        )
        
        # Create LangChain wrapper for the pipeline
        self.llm = HuggingFacePipeline(pipeline=text_generation_pipeline)
        print("Qwen model initialized successfully.")

    def query(self, user_question: str) -> Tuple[str, dict]:
        """Process a user question and return a SPARQL query for Wikidata"""
        response = self.agent_executor.invoke({"input": user_question})
        output = response["output"]
        
        # Extract SPARQL query from the output
        query = output
        
        # If query is wrapped in ```sparql ... ```, extract just the query
        if "```" in query:
            query_parts = query.split("```")
            for i, part in enumerate(query_parts):
                if i % 2 == 1:  # Odd-indexed parts are inside code blocks
                    # Remove "sparql" or other language indicators
                    query = part.strip()
                    if query.lower().startswith("sparql"):
                        query = query[6:].strip()
                    break
        
        # Execute the query to get results
        result = self.sparql_tool._run(query)
        
        return query, result

## 5. Define Evaluation Functions

In [ ]:
import json
import pandas as pd
from tqdm import tqdm
from typing import Dict, List, Tuple

def compare_two_dataframes(df1: pd.DataFrame, df2: pd.DataFrame) -> Dict[str, float]:
    """Compare two dataframes and calculate various metrics."""
    if len(df1.columns) != len(df2.columns):
        return {
            'jaccard': 0,
            'recall': 0,
            'precision': 0,
            'f1': 0,
            'tp': 0,
            'fp': 0,
            'fn': 0,
            'tn': 0
        }

    set1, set2 = set(), set()
    for _, row in df1.iterrows():
        row = list(row)
        row = sorted(row)
        row = tuple(row)
        set1.add(row)

    for _, row in df2.iterrows():
        row = list(row)
        row = sorted(row)
        row = tuple(row)
        set2.add(row)
    
    jaccard = len(set1 & set2) / len(set1 | set2) if len(set1 | set2) > 0 else 0
    recall = len(set1 & set2) / len(set1) if len(set1) > 0 else 0
    precision = len(set1 & set2) / len(set2) if len(set2) > 0 else 0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

    tp = len(set1 & set2)
    fp = len(set2) - tp
    fn = len(set1) - tp
    total_pairs = len(set1) + len(set2) - tp
    tn = total_pairs - (tp + fp + fn)

    return {
        'jaccard': jaccard,
        'recall': recall,
        'precision': precision,
        'f1': f1,
        'tp': tp,
        'fp': fp,
        'fn': fn,
        'tn': tn
    }

def execute_sparql_to_df(query: str) -> pd.DataFrame:
    """Execute a SPARQL query and convert results to DataFrame"""
    sparql_tool = ExecuteSPARQLTool()
    result = sparql_tool._run(query)
    
    if result.get('success', False) and 'results' in result:
        # Convert results to DataFrame
        df = pd.DataFrame(result['results'])
        return df
    else:
        # Return empty DataFrame on error
        return pd.DataFrame()

def evaluate_wikidata_agent(agent: WikidataAgent, test_data_path: str, output_log_path: str = "evaluation_results.json", limit: int = None):
    """Evaluate the Wikidata Agent against a test dataset"""
    
    # Load test data
    with open(test_data_path, 'r') as f:
        test_data = json.load(f)
    
    # Limit the number of test cases if specified
    if limit and limit > 0:
        test_data = test_data[:limit]
    
    # Prepare results storage
    results = []
    metrics_sum = {
        'jaccard': 0,
        'recall': 0,
        'precision': 0,
        'f1': 0,
        'tp': 0,
        'fp': 0,
        'fn': 0,
        'tn': 0
    }
    
    # Process each test question
    for i, test_item in tqdm(enumerate(test_data), total=len(test_data), desc="Evaluating"):
        question = test_item['question']
        ground_truth_query = test_item['sparql']
        
        print(f"\nProcessing question {i+1}/{len(test_data)}: {question}")
        
        try:
            # Generate query using the agent
            generated_query, query_result = agent.query(question)
            
            # Execute both queries to get dataframes
            print("Executing ground truth query...")
            ground_truth_df = execute_sparql_to_df(ground_truth_query)
            
            # Use the result already provided by the agent
            generated_df = pd.DataFrame(query_result.get('results', []))
            
            # Compare results
            metrics = compare_two_dataframes(ground_truth_df, generated_df)
            
            # Update metrics sum
            for key in metrics_sum:
                metrics_sum[key] += metrics[key]
            
            # Save individual result
            result_item = {
                'question': question,
                'ground_truth_query': ground_truth_query,
                'generated_query': generated_query,
                'metrics': metrics,
                'success': True
            }
            
            print(f"Metrics: Precision: {metrics['precision']:.4f}, Recall: {metrics['recall']:.4f}, F1: {metrics['f1']:.4f}")
            
        except Exception as e:
            print(f"Error: {str(e)}")
            result_item = {
                'question': question,
                'ground_truth_query': ground_truth_query,
                'error': str(e),
                'success': False
            }
            
        results.append(result_item)
        
        # Print progress and current average metrics
        if (i + 1) % 5 == 0 or i == len(test_data) - 1:
            avg_metrics = {k: v / (i + 1) for k, v in metrics_sum.items()}
            print(f"\nCurrent average metrics after {i + 1}/{len(test_data)} questions:")
            print(f"Precision: {avg_metrics['precision']:.4f}")
            print(f"Recall: {avg_metrics['recall']:.4f}")
            print(f"F1: {avg_metrics['f1']:.4f}")
            print(f"Jaccard: {avg_metrics['jaccard']:.4f}")
    
    # Calculate average metrics
    avg_metrics = {k: v / len(test_data) for k, v in metrics_sum.items()}
    
    # Save results
    final_results = {
        'results': results,
        'average_metrics': avg_metrics
    }
    
    with open(output_log_path, 'w') as f:
        json.dump(final_results, f, indent=2)
    
    return final_results

## 6. Run Evaluation

In [ ]:
# Initialize the agent
print("Initializing Wikidata Agent...")
agent = WikidataAgent()

# Define paths
test_data_path = "dataset/qald_9_plus/qald_9_plus_test_wikidata.json"
output_log_path = "evaluation_results.json"

# Set a limit for testing (e.g., first 10 questions)
# Remove the limit parameter or set to None to run on all questions
test_limit = 10

# Run evaluation
print(f"Evaluating agent on {test_data_path} (limit: {test_limit})...")
results = evaluate_wikidata_agent(agent, test_data_path, output_log_path, limit=test_limit)

# Print summary
print("\nEvaluation complete!")
print(f"Results saved to {output_log_path}")
print("\nAverage Metrics:")
for key, value in results['average_metrics'].items():
    print(f"{key}: {value:.4f}")

## 7. Analyze Results

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Load results from file if needed
with open(output_log_path, 'r') as f:
    results = json.load(f)

# Extract metrics for successful queries
successful_results = [result for result in results['results'] if result.get('success', False)]
question_ids = [i for i, _ in enumerate(successful_results)]
precision_values = [result['metrics']['precision'] for result in successful_results]
recall_values = [result['metrics']['recall'] for result in successful_results]
f1_values = [result['metrics']['f1'] for result in successful_results]

# Create a DataFrame for easier plotting
metrics_df = pd.DataFrame({
    'Question ID': question_ids,
    'Precision': precision_values,
    'Recall': recall_values,
    'F1 Score': f1_values
})

# Plot metrics
plt.figure(figsize=(14, 7))
sns.set_style("whitegrid")

# Plot metrics by question
plt.subplot(1, 2, 1)
metrics_melted = pd.melt(metrics_df, id_vars=['Question ID'], 
                         value_vars=['Precision', 'Recall', 'F1 Score'],
                         var_name='Metric', value_name='Value')
sns.lineplot(data=metrics_melted, x='Question ID', y='Value', hue='Metric', marker='o')
plt.title('Metrics by Question')
plt.ylim(0, 1.05)

# Plot average metrics
plt.subplot(1, 2, 2)
avg_metrics = results['average_metrics']
metrics_to_plot = ['precision', 'recall', 'f1', 'jaccard']
values_to_plot = [avg_metrics[metric] for metric in metrics_to_plot]
sns.barplot(x=metrics_to_plot, y=values_to_plot)
plt.title('Average Metrics')
plt.ylim(0, 1)

plt.tight_layout()
plt.show()

# Summary table
print("Performance Summary:")
total_questions = len(results['results'])
successful_questions = len(successful_results)
failed_questions = total_questions - successful_questions

summary_df = pd.DataFrame([
    {'Metric': 'Total Questions', 'Value': total_questions},
    {'Metric': 'Successfully Processed', 'Value': successful_questions, 'Percentage': f"{successful_questions/total_questions*100:.1f}%"},
    {'Metric': 'Failed Questions', 'Value': failed_questions, 'Percentage': f"{failed_questions/total_questions*100:.1f}%"},
    {'Metric': 'Average Precision', 'Value': f"{avg_metrics['precision']:.4f}"},
    {'Metric': 'Average Recall', 'Value': f"{avg_metrics['recall']:.4f}"},
    {'Metric': 'Average F1 Score', 'Value': f"{avg_metrics['f1']:.4f}"},
    {'Metric': 'Average Jaccard Similarity', 'Value': f"{avg_metrics['jaccard']:.4f}"}
])

display(summary_df)